# NASA Intelligence Chat System — Project Walkthrough

This notebook walks through the architecture and key design decisions of the NASA RAG system.  
By the end you will understand how every module fits together.

**No API key required** for the non-LLM sections.

---

## Architecture at a glance

```
NASA Text Files
      ↓
  EmbeddingPipeline (nasa_rag.embedding)
      ↓  OpenAI text-embedding-3-small
  ChromaDB vector store
      ↓  cosine similarity search
  retrieve_documents (nasa_rag.retrieval)
      ↓  [Source N] context string
  generate_response (nasa_rag.generation)
      ↓  OpenAI GPT
  Grounded answer with citations
      ↓
  evaluate_response (nasa_rag.evaluation)
      ↓  RAGAS ResponseRelevancy + Faithfulness
  Quality scores
```

---

## 1. Setup

In [1]:
import sys
from pathlib import Path

# Add src/ to Python path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print(f'Project root : {PROJECT_ROOT}')
print(f'src on path  : {SRC_DIR.exists()}')

Project root : /Users/suleimanadebowaleojo/Claude/Projects/NASA Intelligence Chat System
src on path  : True


In [2]:
from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / '.env')

from nasa_rag.config import get_settings, MISSION_NORMALISE, KNOWN_MISSIONS

cfg = get_settings()
print(f'Chat model       : {cfg.chat_model}')
print(f'Embedding model  : {cfg.embedding_model}')
print(f'ChromaDB dir     : {cfg.chroma_dir}')
print(f'Collection       : {cfg.collection_name}')
print(f'Chunk size       : {cfg.chunk_size}')
print(f'Chunk overlap    : {cfg.chunk_overlap}')
print(f'API key set      : {cfg.is_configured()}')

Chat model       : gpt-4o-mini
Embedding model  : text-embedding-3-small
ChromaDB dir     : ./chroma_db
Collection       : nasa_missions
Chunk size       : 1000
Chunk overlap    : 150
API key set      : True


## 2. Module: nasa_rag.config

All runtime parameters live in one place. Nothing is hardcoded across modules.

In [3]:
print('Mission normalisation map:')
for user_input, internal in MISSION_NORMALISE.items():
    print(f'  {user_input!r:20s} → {internal!r}')

Mission normalisation map:
  'apollo 11'          → 'apollo_11'
  'apollo11'           → 'apollo_11'
  'apollo_11'          → 'apollo_11'
  'apollo 13'          → 'apollo_13'
  'apollo13'           → 'apollo_13'
  'apollo_13'          → 'apollo_13'
  'challenger'         → 'challenger'
  'sts-51l'            → 'challenger'
  'sts51l'             → 'challenger'


## 3. Module: nasa_rag.embedding

The `EmbeddingPipeline` class handles document ingestion.  
Let's explore its chunking and metadata extraction logic without making any API calls.

In [4]:
from nasa_rag.embedding import extract_mission, extract_data_type, extract_document_category

# Test the extraction helpers
test_paths = [
    'data_text/apollo11/a11transcript_tec_textract_full_text.txt',
    'data_text/apollo13/AS13_TEC_textract_full_text.txt',
    'data_text/challenger/107-AAG_STS-51L_Mission_Audio_transcript.txt',
]

from pathlib import Path
print(f'{'File':<50} {'Mission':<15} {'Data type':<25} {'Category'}')
print('-' * 110)
for p in test_paths:
    fp = Path(p)
    print(
        f'{fp.name:<50} '
        f'{extract_mission(fp):<15} '
        f'{extract_data_type(fp):<25} '
        f'{extract_document_category(fp.name)}'
    )

File                                               Mission         Data type                 Category
--------------------------------------------------------------------------------------------------------------
a11transcript_tec_textract_full_text.txt           apollo_11       transcript                technical
AS13_TEC_textract_full_text.txt                    apollo_13       textract_extracted        technical
107-AAG_STS-51L_Mission_Audio_transcript.txt       challenger      transcript                mission_audio


In [5]:
# Demonstrate chunking without ChromaDB (mock the DB layer)
from unittest.mock import MagicMock, patch

mock_col = MagicMock()
mock_col.count.return_value = 0

mock_client = MagicMock()
mock_client.get_or_create_collection.return_value = mock_col

with (
    patch('nasa_rag.embedding.chromadb.PersistentClient', return_value=mock_client),
    patch('nasa_rag.embedding.OpenAIEmbeddingFunction', return_value=MagicMock()),
):
    from nasa_rag.embedding import EmbeddingPipeline
    pipeline = EmbeddingPipeline(
        openai_api_key='sk-demo',
        chunk_size=200,
        chunk_overlap=30,
    )

sample_text = (
    "Houston, we've had a problem here. "
    "Okay, Houston, we've had a problem. "
    "We've had a Main B Bus undervolt. "
) * 15  # Repeat to get a long-enough text

chunks = pipeline.chunk_text(sample_text, {'mission': 'apollo_13', 'source': 'demo'})

print(f'Text length  : {len(sample_text)} chars')
print(f'Chunk size   : {pipeline.chunk_size}')
print(f'Chunk overlap: {pipeline.chunk_overlap}')
print(f'Chunks produced: {len(chunks)}')
print()
for i, (text, meta) in enumerate(chunks[:3]):
    print(f'[Chunk {i}] length={len(text)}, index={meta["chunk_index"]}/{meta["total_chunks"]}')
    print(f'  {text[:80]}…')

Text length  : 1575 chars
Chunk size   : 200
Chunk overlap: 30
Chunks produced: 11

[Chunk 0] length=175, index=0/11
  Houston, we've had a problem here. Okay, Houston, we've had a problem. We've had…
[Chunk 1] length=168, index=1/11
  Houston, we've had a problem. We've had a Main B Bus undervolt. Houston, we've h…
[Chunk 2] length=170, index=2/11
  ve had a Main B Bus undervolt. Houston, we've had a problem here. Okay, Houston,…


## 4. Module: nasa_rag.retrieval

The retrieval module wraps ChromaDB with typed helpers.

In [6]:
from nasa_rag.retrieval import format_context

# format_context is a pure function — no DB needed
sample_docs = [
    "The oxygen tank No. 2 ruptured at 55:55 MET, causing a rapid pressure loss.",
    "The crew immediately moved to Lunar Module Aquarius as a lifeboat.",
]
sample_metas = [
    {
        'mission': 'apollo_13',
        'file_path': 'data_text/apollo13/AS13_TEC_textract_full_text.txt',
        'document_category': 'technical',
        'chunk_index': 4,
        'total_chunks': 92,
        'source': 'AS13_TEC_textract_full_text',
    },
    {
        'mission': 'apollo_13',
        'file_path': 'data_text/apollo13/AS13_PAO_textract_full_text.txt',
        'document_category': 'public_affairs_officer',
        'chunk_index': 7,
        'total_chunks': 55,
        'source': 'AS13_PAO_textract_full_text',
    },
]

context = format_context(sample_docs, sample_metas)
print(context)

=== RETRIEVED NASA MISSION CONTEXT ===

[Source 1]
Mission: Apollo 13
File: data_text/apollo13/AS13_TEC_textract_full_text.txt
Category: Technical
Chunk: 4/92
Content:
The oxygen tank No. 2 ruptured at 55:55 MET, causing a rapid pressure loss.

[Source 2]
Mission: Apollo 13
File: data_text/apollo13/AS13_PAO_textract_full_text.txt
Category: Public Affairs Officer
Chunk: 7/55
Content:
The crew immediately moved to Lunar Module Aquarius as a lifeboat.



## 5. Module: nasa_rag.generation

The `generate_response` function builds a multi-turn message list and calls OpenAI.

In [7]:
from nasa_rag.generation import _SYSTEM_PROMPT, _CONTEXT_PREFIX, _NO_CONTEXT_MESSAGE

print('=== SYSTEM PROMPT ===')
print(_SYSTEM_PROMPT)
print()
print('=== NO CONTEXT MESSAGE ===')
print(_NO_CONTEXT_MESSAGE)

=== SYSTEM PROMPT ===
You are a NASA mission intelligence assistant. Your role is to answer questions using ONLY the retrieved NASA mission context provided to you.

Guidelines:
1. Answer clearly and accurately based solely on the provided context.
2. Always cite your sources using the [Source N] labels from the context    (e.g. "According to [Source 1] — Apollo 13 Technical Transcript …").
3. If the retrieved context does not contain enough evidence to answer the    question, say so clearly:
   "The available NASA documents do not provide sufficient information to answer this question."
4. Never fabricate facts, dates, names, or technical details not present in    the context.
5. When relevant, mention the mission name, document type (transcript, technical    report, flight plan), and any key identifiers.
6. Keep answers focused and grounded; prefer evidence over speculation.


=== NO CONTEXT MESSAGE ===
No relevant documents were retrieved from the NASA archive for this query. Inform

## 6. Data corpus overview

In [8]:
import os

data_dir = PROJECT_ROOT / 'data_text'
missions = ['apollo11', 'apollo13', 'challenger']

print(f'{'Mission':<15} {'Files':<8} {'Total size (KB)'}')
print('-' * 40)

for mission in missions:
    mission_dir = data_dir / mission
    if mission_dir.exists():
        files = list(mission_dir.glob('*.txt'))
        total_bytes = sum(f.stat().st_size for f in files)
        print(f'{mission:<15} {len(files):<8} {total_bytes / 1024:.1f}')
    else:
        print(f'{mission:<15} (directory not found)')

Mission         Files    Total size (KB)
----------------------------------------
apollo11        6        3278.1
apollo13        3        2431.3
challenger      3        371.3


## 7. Run the test suite

You can validate the entire codebase without opening a terminal.

In [9]:
import sys
import subprocess
result = subprocess.run(
    [sys.executable, '-m', 'pytest', 'tests/', '-v', '--tb=short'],
    cwd=str(PROJECT_ROOT),
    capture_output=True,
    text=True,
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:')
    print(result.stderr)

============================= test session starts ==============================
platform darwin -- Python 3.13.5, pytest-8.3.4, pluggy-1.6.0
rootdir: /Users/suleimanadebowaleojo/Claude/Projects/NASA Intelligence Chat System
configfile: pyproject.toml
plugins: mock-3.15.1, cov-5.0.0, anyio-4.7.0
collected 96 items

tests/test_config.py .................                                   [ 17%]
tests/test_embedding.py ................................                 [ 51%]
tests/test_evaluation.py .........                                       [ 60%]
tests/test_generation.py ...............                                 [ 76%]
tests/test_retrieval.py .......................                          [100%]

============================== 96 passed in 3.93s ==============================

